## Autoencoder Latent Space Plot


### Example: Simple Autoencoder (PyTorch)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# -------------------
# 1. Dataset
# -------------------
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# -------------------
# 2. Autoencoder
# -------------------
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )
    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

latent_dim = 2  # for easy 2D visualization
model = Autoencoder(latent_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------
# 3. Train Autoencoder (short example)
# -------------------
for epoch in range(5):
    for imgs, _ in train_loader:
        optimizer.zero_grad()
        outputs, _ = model(imgs)
        loss = criterion(outputs, imgs)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


ModuleNotFoundError: No module named 'torchvision'

### Extract Latent Representations

In [2]:
all_latents = []
all_labels = []

with torch.no_grad():
    for imgs, labels in train_loader:
        _, z = model(imgs)
        all_latents.append(z)
        all_labels.append(labels)

all_latents = torch.cat(all_latents).numpy()
all_labels = torch.cat(all_labels).numpy()


NameError: name 'train_loader' is not defined

### 2D Latent Space Plot

In [3]:
plt.figure(figsize=(8,6))
for label in np.unique(all_labels):
    plt.scatter(
        all_latents[all_labels==label, 0],
        all_latents[all_labels==label, 1],
        label=f"Digit {label}", alpha=0.6
    )
plt.xlabel("Latent Dim 1")
plt.ylabel("Latent Dim 2")
plt.title("2D Latent Space of Autoencoder")
plt.legend()
plt.show()


NameError: name 'plt' is not defined

### 3D Latent Space (if latent_dim > 2)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

latent_dim = 3  # for 3D visualization
# Rebuild model.encoder with latent_dim=3 if needed
# Use same extraction method as above

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
for label in np.unique(all_labels):
    ax.scatter(
        all_latents[all_labels==label,0],
        all_latents[all_labels==label,1],
        all_latents[all_labels==label,2],
        label=f"Digit {label}", alpha=0.6
    )
ax.set_title("3D Latent Space")
plt.legend()
plt.show()


### Optional: t-SNE / UMAP on Latent Space (Useful if latent_dim > 3)

In [ ]:
from sklearn.manifold import TSNE
import umap

# t-SNE
X_tsne = TSNE(n_components=2, random_state=42).fit_transform(all_latents)

plt.figure(figsize=(7,5))
for label in np.unique(all_labels):
    plt.scatter(X_tsne[all_labels==label,0], X_tsne[all_labels==label,1], label=f"Digit {label}", alpha=0.6)
plt.title("t-SNE of Autoencoder Latent Space")
plt.legend()
plt.show()

# UMAP
X_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(all_latents)

plt.figure(figsize=(7,5))
for label in np.unique(all_labels):
    plt.scatter(X_umap[all_labels==label,0], X_umap[all_labels==label,1], label=f"Digit {label}", alpha=0.6)
plt.title("UMAP of Autoencoder Latent Space")
plt.legend()
plt.show()
